# Step 3: Embed Proposals

This notebook computes multilingual embeddings for all proposals using E5, Qwen, BERT, and Qwen-128 (Matryoshka) models.

In [ ]:
# © Copyright European Union - 2026

import os
import json
import re
import random
import glob
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"]
)

MODEL_NAME = "llama-3.3-70b-instruct"

random.seed(42)


def extract_json_from_reply(raw_output: str):
    """Extract JSON from LLM reply text."""
    try:
        return json.loads(raw_output)
    except (json.JSONDecodeError, TypeError):
        pass
    raw_output = raw_output.strip()
    if "```json" in raw_output:
        raw_output = raw_output.split("```json")[-1].split("```")[0]
    elif "```" in raw_output:
        parts = raw_output.split("```")
        if len(parts) >= 3:
            raw_output = parts[1]
    # Find JSON structure
    for start_char, end_char in [("[", "]"), ("{", "}")]:
        if start_char in raw_output and end_char in raw_output:
            start = raw_output.index(start_char)
            end = raw_output.rindex(end_char) + 1
            try:
                return json.loads(raw_output[start:end])
            except json.JSONDecodeError:
                continue
    return json.loads(raw_output)


def model_json(system_prompt: str, user_message: str) -> dict:
    """Call LLM and return parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return extract_json_from_reply(response.choices[0].message.content)


def model(system_prompt: str, user_message: str) -> str:
    """Call LLM and return raw text response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return response.choices[0].message.content


In [ ]:
emb = client.embeddings.create(
        model="multilingual-e5-large",
        input="Ciao"
    )
print(np.array(emb.data[0].embedding, np.float32))

In [ ]:
DATA_OUT_BASE_PATH = "../out/"
chunk_clusters_fp = f"{DATA_OUT_BASE_PATH}2.chunk_clusters.json"
merged_clusters_fp = f"{DATA_OUT_BASE_PATH}2.merged_clusters.json"
classified_proposals_fp = f"{DATA_OUT_BASE_PATH}3.classified_proposals.json"
classified_proposals_errors_fp = f"{DATA_OUT_BASE_PATH}3.classified_proposals_errors.json"
clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}3.clusterized_proposals.json"
clusterized_proposals_with_embeddings_fp = f"{DATA_OUT_BASE_PATH}4.clusterized_proposals_with_embeddings.json"
proposal_embeddings_fp = "../out/proposal_embeddings.json"
list_proposals = json.load(open("../data/proposals.json"))

In [ ]:
def chunked_iterator(l, chunk_size=100):
    """Yield chunks of the DataFrame."""
    for start in range(0, len(l), chunk_size):
        yield l[start:start+chunk_size]


In [ ]:
def embed_with_gptatjrc(gptatjrc_client, proposals):
    """Embed proposals using the E5 multilingual model via OpenAI-compatible API."""
    embeddings = []
    chunk_embeddings = gptatjrc_client.embeddings.create(
        model="multilingual-e5-large",
        input=proposals
    )
    for embedding in chunk_embeddings.data:
        embeddings.append(embedding.embedding)
    return embeddings


In [ ]:
from sentence_transformers import SentenceTransformer
def embed_with_sentence_transformer(embedder, proposals):
    embeddings = []
    embeddings = embedder.encode(proposals)
    embeddings = [x.tolist() for x in embeddings]
    return embeddings


In [ ]:
qwen_model = SentenceTransformer("Qwen/Qwen3-Embedding-8B", trust_remote_code=True)
bert_model = SentenceTransformer("distiluse-base-multilingual-cased-v2", trust_remote_code=True)

In [ ]:
chunk_size = 32
proposal_embeddings = {}

proposal_chunks = [x for x in chunked_iterator(list_proposals, chunk_size)]

for chunk in tqdm(proposal_chunks, desc="Proposals chunks"):

    chunk_titles = [x['title'] for x in chunk]
    chunk_embeddings_e5 = embed_with_gptatjrc(client, chunk_titles)
    chunk_embeddings_qween = embed_with_sentence_transformer(qwen_model, chunk_titles)
    chunk_embeddings_bert = embed_with_sentence_transformer(bert_model, chunk_titles)

    # 
    for idx, embedding_e5 in enumerate(chunk_embeddings_e5):
        proposal_embeddings[chunk[idx]['guid']] = {
            'embedding_e5': embedding_e5,
            'embedding_qwen': chunk_embeddings_qween[idx],
            'embedding_bert': chunk_embeddings_bert[idx]
        }

json.dump(proposal_embeddings, open(proposal_embeddings_fp, 'w'), ensure_ascii=False)

### Add embedding type

In [ ]:
list_proposals = json.load(open("../data/proposals.json"))
proposal_embeddings = json.load(open(proposal_embeddings_fp))

In [ ]:
p = list(proposal_embeddings.values())[0]
print("Qwen dimensions:", len(p["embedding_qwen"]))
if "embedding_qwen_128" in p:
    print("Qwen_128 dimensions:", len(p["embedding_qwen_128"]))
if "embedding_bert" in p:
    print("BERT dimensions:", len(p["embedding_bert"]))
print("E5 dimensions:", len(p["embedding_e5"]))


In [ ]:
#new_model = SentenceTransformer("distiluse-base-multilingual-cased-v2", trust_remote_code=True)
# Test with Matryoshka embeddings as MRL is supported by Qwen3-Embeddings-8B - https://github.com/QwenLM/Qwen3-Embedding
new_model = SentenceTransformer("Qwen/Qwen3-Embedding-8B", trust_remote_code=True, truncate_dim=128)

In [ ]:
chunk_size = 32

proposal_chunks = [x for x in chunked_iterator(list_proposals, chunk_size)]

for chunk in tqdm(proposal_chunks, desc="Proposals chunks"):

    chunk_titles = [x['title'] for x in chunk]
    embeddings = embed_with_sentence_transformer(new_model, chunk_titles)

    # 
    for idx, embedding in enumerate(embeddings):
        proposal_embeddings[chunk[idx]['guid']]['embedding_qwen_128'] = embedding

json.dump(proposal_embeddings, open(proposal_embeddings_fp, 'w'), ensure_ascii=False)